# Query 6: Trips Above the Average Fare Per Pickup Area
**Type:** Subquery / Nested Query
**Problem:** Find trips whose fare exceeds the average fare of their pickup area (rounded lat/lon grid cell).

In [1]:
import time
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName('Q6_Subquery') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

26/04/25 17:58:01 WARN Utils: Your hostname, mariam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/25 17:58:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 17:58:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


In [2]:
DATA_PATH = '../data/yellow_tripdata_2015-01.csv'

df = spark.read.option('header','true').option('inferSchema','true').csv(DATA_PATH)

# Create a pickup_zone by rounding coordinates to 2 decimal places (grid cell)
df = df.withColumn('pickup_zone',
        F.concat(
            F.round(F.col('pickup_latitude'), 2).cast('string'),
            F.lit('_'),
            F.round(F.col('pickup_longitude'), 2).cast('string')
        )) \
       .withColumnRenamed('fare_amount', 'fare') \
       .withColumnRenamed('passenger_count', 'passengers')

df.createOrReplaceTempView('trips')
rdd = df.rdd
print('Total rows:', df.count())
df.select('pickup_zone','fare','passengers').show(5)

Total rows: 11157879
+------------+----+----------+
| pickup_zone|fare|passengers|
+------------+----+----------+
|40.75_-73.99|12.0|       1.0|
| 40.72_-74.0|14.5|       1.0|
| 40.8_-73.96| 9.5|       1.0|
|40.71_-74.01| 3.5|       1.0|
|40.76_-73.97|15.0|       1.0|
+------------+----+----------+
only showing top 5 rows



## RDD Implementation

In [3]:
start = time.time()

zone_stats = (
    rdd
    .filter(lambda r: r['pickup_zone'] is not None and r['fare'] is not None)
    .map(lambda r: (r['pickup_zone'], (float(r['fare']), 1)))
    .reduceByKey(lambda a, b: (a[0]+b[0], a[1]+b[1]))
    .mapValues(lambda x: x[0]/x[1])
)
avg_map = dict(zone_stats.collect())
avg_bc  = spark.sparkContext.broadcast(avg_map)

result_rdd = rdd.filter(
    lambda r: r['pickup_zone'] is not None
    and r['fare'] is not None
    and r['pickup_zone'] in avg_bc.value
    and float(r['fare']) > avg_bc.value[r['pickup_zone']]
)

rdd_count = result_rdd.count()
rdd_time  = time.time() - start
print(f'RDD  | Trips above zone avg: {rdd_count:,} | Time: {rdd_time:.2f}s')

RDD  | Trips above zone avg: 3,942,347 | Time: 239.16s


## DataFrame Implementation

In [4]:
start = time.time()

zone_avg = df.groupBy('pickup_zone').agg(F.avg('fare').alias('zone_avg_fare'))

result_df = (
    df.join(zone_avg, on='pickup_zone', how='inner')
      .filter(F.col('fare') > F.col('zone_avg_fare'))
      .select('pickup_zone','fare','zone_avg_fare','tpep_pickup_datetime','payment_type')
)
result_df.explain(True)
df_count = result_df.count()
df_time  = time.time() - start
print(f'DataFrame | Trips above zone avg: {df_count:,} | Time: {df_time:.2f}s')
result_df.show(10)

== Parsed Logical Plan ==
'Project ['pickup_zone, 'fare, 'zone_avg_fare, 'tpep_pickup_datetime, 'payment_type]
+- Filter (fare#77 > zone_avg_fare#204)
   +- Project [pickup_zone#55, VendorID#17, tpep_pickup_datetime#18, tpep_dropoff_datetime#19, passengers#98, trip_distance#21, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#77, extra#30, mta_tax#31, tip_amount#32, tolls_amount#33, improvement_surcharge#34, total_amount#35, zone_avg_fare#204]
      +- Join Inner, (pickup_zone#55 = pickup_zone#226)
         :- Project [VendorID#17, tpep_pickup_datetime#18, tpep_dropoff_datetime#19, passenger_count#20 AS passengers#98, trip_distance#21, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#77, extra#30, mta_tax#31, tip_amount#32, tolls_amount#33, improvement_surcharge#34, total_amount#35, pickup_zone#55]


DataFrame | Trips above zone avg: 3,942,347 | Time: 36.09s


+------------+----+------------------+--------------------+------------+
| pickup_zone|fare|     zone_avg_fare|tpep_pickup_datetime|payment_type|
+------------+----+------------------+--------------------+------------+
|40.75_-73.98|17.5|10.391520995051494| 2015-01-15 19:05:41|         2.0|
| 40.67_-73.8|52.0| 49.82231818181818| 2015-01-15 14:00:43|         1.0|
|40.75_-73.98|10.5|10.391520995051494| 2015-01-15 14:00:46|         1.0|
|40.74_-73.97|19.0|10.669319908814593| 2015-01-15 14:00:47|         2.0|
|40.74_-73.97|15.0|10.669319908814593| 2015-01-26 12:41:12|         1.0|
|40.76_-73.96|16.0|10.369860839819543| 2015-01-15 10:26:14|         1.0|
|40.75_-73.98|11.0|10.391520995051494| 2015-01-18 19:49:31|         1.0|
|40.81_-73.96|24.0| 12.12995023582278| 2015-01-28 10:50:05|         1.0|
|40.75_-73.98|14.5|10.391520995051494| 2015-01-07 20:09:42|         1.0|
|40.74_-73.97|17.5|10.669319908814593| 2015-01-13 00:09:41|         2.0|
+------------+----+------------------+-------------

## Spark SQL Implementation

In [5]:
start = time.time()

result_sql = spark.sql("""
    SELECT t.pickup_zone, t.fare, z.zone_avg_fare,
           t.tpep_pickup_datetime, t.payment_type
    FROM   trips t
    JOIN   (
               SELECT pickup_zone, AVG(fare) AS zone_avg_fare
               FROM   trips
               GROUP BY pickup_zone
           ) z ON t.pickup_zone = z.pickup_zone
    WHERE  t.fare > z.zone_avg_fare
    ORDER BY t.fare DESC
""")
result_sql.explain(True)
sql_count = result_sql.count()
sql_time  = time.time() - start
print(f'SQL | Trips above zone avg: {sql_count:,} | Time: {sql_time:.2f}s')
result_sql.show(10)

== Parsed Logical Plan ==
'Sort ['t.fare DESC NULLS LAST], true
+- 'Project ['t.pickup_zone, 't.fare, 'z.zone_avg_fare, 't.tpep_pickup_datetime, 't.payment_type]
   +- 'Filter ('t.fare > 'z.zone_avg_fare)
      +- 'Join Inner, ('t.pickup_zone = 'z.pickup_zone)
         :- 'SubqueryAlias t
         :  +- 'UnresolvedRelation [trips], [], false
         +- 'SubqueryAlias z
            +- 'Aggregate ['pickup_zone], ['pickup_zone, 'AVG('fare) AS zone_avg_fare#319]
               +- 'UnresolvedRelation [trips], [], false

== Analyzed Logical Plan ==
pickup_zone: string, fare: double, zone_avg_fare: double, tpep_pickup_datetime: timestamp, payment_type: double
Sort [fare#77 DESC NULLS LAST], true
+- Project [pickup_zone#55, fare#77, zone_avg_fare#319, tpep_pickup_datetime#18, payment_type#28]
   +- Filter (fare#77 > zone_avg_fare#319)
      +- Join Inner, (pickup_zone#55 = pickup_zone#341)
         :- SubqueryAlias t
         :  +- SubqueryAlias trips
         :     +- View (`trips`, [VendorI

SQL | Trips above zone avg: 3,942,347 | Time: 29.83s


+------------+------+------------------+--------------------+------------+
| pickup_zone|  fare|     zone_avg_fare|tpep_pickup_datetime|payment_type|
+------------+------+------------------+--------------------+------------+
|40.71_-74.01|3005.5|14.878546768038696| 2015-01-02 20:06:34|         2.0|
|     0.0_0.0|999.99|12.945360364379127| 2015-01-23 11:15:00|         1.0|
|     0.0_0.0|999.99|12.945360364379127| 2015-01-16 14:48:00|         1.0|
|40.78_-73.92| 980.0|17.435494880546077| 2015-01-28 08:54:07|         1.0|
|40.69_-73.81| 965.0|50.171648000000005| 2015-01-17 05:54:37|         1.0|
|40.69_-73.59| 965.0|            678.33| 2015-01-09 17:42:08|         1.0|
|40.69_-73.59|949.99|            678.33| 2015-01-07 13:00:53|         1.0|
|40.78_-73.91| 900.0|13.203623096446703| 2015-01-20 07:07:40|         1.0|
|40.72_-73.99| 900.0|11.825941771887736| 2015-01-08 11:47:20|         1.0|
|40.72_-73.99| 900.0|11.825941771887736| 2015-01-07 16:34:56|         1.0|
+------------+------+----

## Performance Comparison

In [6]:
print('='*60)
print(f'{"Metric":<20} {"RDD":>12} {"DataFrame":>12} {"SQL":>12}')
print('-'*60)
print(f'{"Execution Time":<20} {rdd_time:>11.2f}s {df_time:>11.2f}s {sql_time:>11.2f}s')
print(f'{"Result Count":<20} {rdd_count:>12,} {df_count:>12,} {sql_count:>12,}')
print(f'{"Optimizer":<20} {"None":>12} {"Catalyst":>12} {"Catalyst":>12}')
print('='*60)

Metric                        RDD    DataFrame          SQL
------------------------------------------------------------
Execution Time            239.16s       36.09s       29.83s
Result Count            3,942,347    3,942,347    3,942,347
Optimizer                    None     Catalyst     Catalyst
